# A4 欧氏 GCN 与 A5 类别平衡 GCN

独立 kernel、单卡顺序运行两组；默认 3 个切片 × 5 个 seed。完整协议见 [README](README.md#a0a5-欧氏对照实验)。本轮是探索性表示实验，不使用隐藏切片选参。


In [ ]:
# 必须在导入 torch 前运行；切换显卡或更新代码后重启 kernel。
import os
import sys
GPU_ID = "0"  # 调度器管理可见卡时设为 None。
if "torch" in sys.modules:
    raise RuntimeError("请重启独立 kernel，并从第一格运行。")
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", "scheduler/default"))


In [ ]:
from pathlib import Path
from copy import deepcopy
PROJECT = next((candidate for parent in (Path.cwd(), *Path.cwd().parents)
                for candidate in (parent, parent / "MMDeepSpatial")
                if (candidate / "mmdeepspatial/euclidean.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("请在 MMDeepSpatial 或其父目录打开 notebook。")
sys.path.insert(0, str(PROJECT))
import pandas as pd
from mmdeepspatial.euclidean import EUCLIDEAN_CONFIG, run_arm, compare_arms
from mmdeepspatial.data import input_paths
print("Project:", PROJECT)


## 统一配置

三个 notebook 的科学配置须一致。A3/A5 使用训练标签计算重建损失权重，probe 设置不变。GCN 只用可见同切片表达构图，新细胞仅连接冻结训练参考库，不使用隐藏 XY 或查询之间的边。

若做短流程检查，统一改成一个 holdout、一个 seed、两轮训练，并将 `REUSE_A2_ROOT=None`，另用测试输出目录。正式运行恢复全部配置。


In [ ]:
ARMS_TO_RUN = ['A4', 'A5']
DATA_GLOB = str(PROJECT.parent / "data/merfish_mouse_hypothalamus/**/merfish_[0-4].h5ad")
OUTPUT_ROOT = PROJECT / "outputs/euclidean_v1"
HOLDOUTS = [1, 2, 3]
SEEDS = [42, 0, 1, 2, 3]
EPOCHS = 100
DIM = 32
HIDDEN = 128
GRAPH_K = 10
BALANCE_POWER = 0.5  # 逆平方根频率，样本权重全训练集均值为 1。
LABEL_KEY = "cell_class"
LAYER = None
TRANSFORM = "log1p"  # X 已 log / 标准化时统一改 identity。
DEVICE = "cuda"
# 可复用上一轮兼容的 E0 AE；设为 None 则重新训练 A2。
REUSE_A2_ROOT = PROJECT / "outputs/representations_v2/E0_ae"
if not REUSE_A2_ROOT.is_dir():
    REUSE_A2_ROOT = None
CONFIG = deepcopy(EUCLIDEAN_CONFIG)
CONFIG.update(input_glob=DATA_GLOB, output_root=str(OUTPUT_ROOT), holdouts=HOLDOUTS,
              seeds=SEEDS, epochs=EPOCHS, dim=DIM, hidden=HIDDEN, graph_k=GRAPH_K,
              balance_power=BALANCE_POWER, label_key=LABEL_KEY, layer=LAYER,
              transform=TRANSFORM, device=DEVICE,
              reuse_a2_root=str(REUSE_A2_ROOT) if REUSE_A2_ROOT else None)
CONFIG


In [ ]:
paths = input_paths(CONFIG["input_glob"])
display(pd.DataFrame({"slice_index": range(len(paths)), "file": [str(p) for p in paths],
                      "held_out": [i in HOLDOUTS for i in range(len(paths))]}))
print("Arms:", ARMS_TO_RUN, "runs per arm:", len(HOLDOUTS)*len(SEEDS))
print("A0/A1 不训练神经编码器；A2 可复用已核验的旧权重。")


In [ ]:
results = pd.concat([run_arm(arm, CONFIG) for arm in ARMS_TO_RUN], ignore_index=True)
display(results[["arm", "holdout", "seed", "encoder_uses_labels", "encoder_reused",
                 "rare_macro_auprc", "ultra_rare_macro_auprc", "low_rare_macro_auprc",
                 "rare_macro_f1", "rare_macro_retrieval_hit_at_5", "common_macro_f1",
                 "trainable_parameters"]])


In [ ]:
metrics = ["rare_macro_auprc", "ultra_rare_macro_auprc", "low_rare_macro_auprc",
           "rare_macro_f1", "rare_macro_recall", "rare_macro_retrieval_hit_at_5",
           "rare_macro_knn_purity_at_15", "common_macro_f1"]
by_fold = results.groupby(["arm", "holdout"])[metrics].mean()
display(by_fold)
# 描述性切片间统计，不是独立生物学重复的置信区间。
display(by_fold.groupby("arm").agg(["mean", "std"]))


## 六组全部完成后统一比较

任何一个 notebook 将下方开关改为 True 后运行。比较会核对完整 seed/split、输入、预处理、源码、容量和 A4/A5 图一致性。保存六组原始指标、稀有度分层、逐类表及预定义配对差值，不自动选择最佳模型。


In [ ]:
COMPARE_WHEN_READY = False
if COMPARE_WHEN_READY:
    all_runs, aggregate, all_types = compare_arms(OUTPUT_ROOT)
    display(aggregate)
    display(pd.read_csv(OUTPUT_ROOT / "comparison/paired_deltas.csv"))
